**DATA PREPARATION**

In [1]:
# Importing libraries
import pandas as pd
import numpy as np

# STEP 1: Now what we do is loading of all the data we are supposed to use 

# Credit snapshots
snap_files = {
    'jan': './datasets/Credit Data - 01-01-2025.csv',
    'mar': './datasets/Credit Data - 30-03-2025.csv',
    'jun': './datasets/Credit Data - 30-06-2025.csv',
    'sep': './datasets/Credit Data - 30-09-2025.csv',
    'dec': './datasets/Credit Data - 30-12-2025.csv'
}
snap_dates = {
    'jan': pd.Timestamp('2025-01-01'),
    'mar': pd.Timestamp('2025-03-30'),
    'jun': pd.Timestamp('2025-06-30'),
    'sep': pd.Timestamp('2025-09-30'),
    'dec': pd.Timestamp('2025-12-30'),
}

# Load and stack all snapshots into one table
snaps = []
for name, path in snap_files.items():
    df = pd.read_csv(path)
    df['snapshot'] = name
    df['snapshot_date'] = snap_dates[name]
    snaps.append(df)
credit = pd.concat(snaps, ignore_index=True)

# STEP 2: Load demographics(Date of Birth, Income Level, Gender, Sales Details)

xl_path = './datasets/Sales and Customer Data.xlsx'

dob_df = pd.read_excel(xl_path, sheet_name='DOB').dropna(how='all')
income_df = pd.read_excel(xl_path, sheet_name='Income Level').dropna(how='all')
gender_df = pd.read_excel(xl_path, sheet_name='Gender').dropna(how='all')
sales_df = pd.read_excel(xl_path, sheet_name='Sales Details').dropna(how='all')

# Fix DOB column name (has trailing space)
dob_df = dob_df.rename(columns={'Loan Id ': 'Loan Id'})

# STEP 3: Clean DOB and calculate age

# Parse date — strip timezone info
dob_df['dob'] = pd.to_datetime(dob_df['date_of_birth'], errors='coerce', utc=True)
dob_df['dob'] = dob_df['dob'].dt.tz_localize(None)

# Drop rows with no Loan Id (can't link them)
dob_df = dob_df.dropna(subset=['Loan Id'])

# Keep only most recent DOB record per loan (in case of duplicates)
dob_df = dob_df.sort_values('createdAt UTC', ascending=False)
dob_clean = dob_df.drop_duplicates(subset='Loan Id', keep='first')[['Loan Id','dob']]

print(f"DOB records with valid Loan Id: {len(dob_clean):,}")
print(f"DOB records with valid date: {dob_clean['dob'].notnull().sum():,}")

# STEP 4: Age calculation function 

def calculate_age(dob, snapshot_date):
    if pd.isnull(dob):
        return np.nan
    age = snapshot_date.year - dob.year
    if (snapshot_date.month, snapshot_date.day) < (dob.month, dob.day):
        age -= 1
    return age

#STEP 5: Apply age per snapshot

for snap_name, snap_date in snap_dates.items():
    dob_clean[f'age_{snap_name}'] = dob_clean['dob'].apply(
        lambda d: calculate_age(d, snap_date)
    )

#  STEP 6: Flag impossible ages
# Under 18 or over 80 = data quality issue, exclude from analysis

for snap_name in snap_dates.keys():
    col = f'age_{snap_name}'
    invalid = (dob_clean[col] < 18) | (dob_clean[col] > 80)
    dob_clean.loc[invalid, col] = np.nan  # set to null, don't delete

print(f"\nInvalid ages excluded (under 18 or over 80): {invalid.sum()}")

#STEP 7: Create age bands

age_bins   = [17, 25, 35, 45, 55, 999]
age_labels = ['18-25', '26-35', '36-45', '46-55', '55+']

for snap_name in snap_dates.keys():
    dob_clean[f'age_band_{snap_name}'] = pd.cut(
        dob_clean[f'age_{snap_name}'],
        bins=age_bins,
        labels=age_labels,
        right=True
    )

print("\nAge band distribution (as at Jan 2025):")
print(dob_clean['age_band_jan'].value_counts().sort_index())

#STEP 8: Average monthly income 

income_df = income_df.dropna(subset=['Loan Id'])
income_df['avg_monthly_income'] = income_df['Received'] / income_df['Duration']

# Remove impossible values
income_df.loc[income_df['avg_monthly_income'] < 0, 'avg_monthly_income'] = np.nan
income_df.loc[income_df['avg_monthly_income'] > 5_000_000, 'avg_monthly_income'] = np.nan

income_bins   = [0, 4999, 9999, 19999, 29999, 49999, 99999, 149999, float('inf')]
income_labels = ['Below 5K','5K-9.9K','10K-19.9K','20K-29.9K',
                 '30K-49.9K','50K-99.9K','100K-149.9K','150K+']

income_df['income_band'] = pd.cut(
    income_df['avg_monthly_income'],
    bins=income_bins, labels=income_labels, right=True
)

print("\nIncome band distribution:")
print(income_df['income_band'].value_counts().sort_index())
# STEP 9: Build main(joined) demographics table

demographics = dob_clean.merge(
    income_df[['Loan Id','avg_monthly_income','income_band']],
    on='Loan Id', how='left'
).merge(
    gender_df[['Loan Id','Gender']],
    on='Loan Id', how='left'
)

print(f"\nDemographics master table: {len(demographics):,} rows")
print(f"With income data: {demographics['avg_monthly_income'].notnull().sum():,}")
print(f"With gender data: {demographics['Gender'].notnull().sum():,}")

# STEP 10: Merge into credit snapshots 

master = credit.merge(
    demographics.rename(columns={'Loan Id': 'LOAN_ID'}),
    on='LOAN_ID', how='left'
)

# Pick the right age band per row based on snapshot
master['age_band'] = master.apply(
    lambda row: row.get(f"age_band_{row['snapshot']}", np.nan), axis=1
)

# Check match rates
total = len(credit)
with_age = master['dob'].notnull().sum()
with_income = master['avg_monthly_income'].notnull().sum()
print(f"\n MAIN TABLE MATCH RATES ")
print(f"Total credit records: {total:,}")
print(f"Records with age data: {with_age:,} ({with_age/total:.1%})")
print(f"Records with income data: {with_income:,} ({with_income/total:.1%})")

DOB records with valid Loan Id: 11,217
DOB records with valid date: 11,175

Invalid ages excluded (under 18 or over 80): 4

Age band distribution (as at Jan 2025):
age_band_jan
18-25    3159
26-35    5637
36-45    1777
46-55     502
55+        97
Name: count, dtype: int64

Income band distribution:
income_band
Below 5K        301
5K-9.9K         696
10K-19.9K      1634
20K-29.9K      1461
30K-49.9K      1953
50K-99.9K      2702
100K-149.9K    1238
150K+          1881
Name: count, dtype: int64

Demographics master table: 17,867 rows
With income data: 16,597
With gender data: 16,404

 MAIN TABLE MATCH RATES 
Total credit records: 71,456
Records with age data: 42,677 (59.7%)
Records with income data: 38,462 (53.8%)


**Question 1: Portfolio Health (40%)**

Pick 3–5 metrics you would use to track credit portfolio performance over time. Show how they trend
across the snapshots and highlight one segment (age or income) where risk behaviour differs
meaningfully from the portfolio average. Explain why this matters operationally.

In [2]:
import pandas as pd
import numpy as np

# STEP 1: LOAD AND STACK ALL SNAPSHOTS 
# We combine all 5 files into ONE big table
# Each row gets a 'snapshot' label so we know which photo it came from

snaps = []
for name, path in snap_files.items():
    df = pd.read_csv(path)
    df['snapshot'] = name          # tag each row with its period
    df['snapshot_date'] = snap_dates[name]  # actual date
    snaps.append(df)

credit = pd.concat(snaps, ignore_index=True)
# Result: one table with 71,456 rows (all 5 snapshots combined)

In [3]:
# STEP 2: DEFINE "ACTIVE" LOANS 
# This is critical. We EXCLUDE Paid Off and Returned accounts
# from the denominator when calculating risk rates.
# Why? Because a paid-off loan is a SUCCESS including it would
# make the portfolio look artificially healthy.

active_mask = ~df['ACCOUNT_STATUS_L2'].isin(['Paid Off', 'Return'])
active_df = df[active_mask]

# The ~ means "NOT" — so active = everything that is NOT Paid Off or Return

In [4]:
# STEP3: calculating PAR30 
active_df = credit[~credit['ACCOUNT_STATUS_L2'].isin(['Paid Off', 'Return'])]
n_active = len(active_df)

par30 = (active_df['DAYS_PAST_DUE'] > 30).sum()
par30_rate = par30 / n_active
print(f"PAR30 Rate: {par30_rate:.1%}")

PAR30 Rate: 43.2%


In [5]:
# STEP 4: LOAN AGE BANDING 
# CUSTOMER_AGE = days since sale (we confirmed this earlier)
# We group loans into age buckets using pd.cut

loan_age_bins   = [0, 90, 180, 365, 730, 9999]
loan_age_labels = ['0-3 months', '3-6 months', '6-12 months', '1-2 years', '2+ years']
#                   ↑ 90 days = 3 months, 180 = 6 months, 365 = 1 year etc.

credit['loan_age_band'] = pd.cut(
    credit['CUSTOMER_AGE'],
    bins=loan_age_bins,
    labels=loan_age_labels,
    right=True   # (0–90] means up to AND including 90
)

In [6]:
# STEP 5 Computing metrics by Segment
# define active_dec first
dec = credit[credit['snapshot'] == 'dec'].copy()
active_dec = dec[~dec['ACCOUNT_STATUS_L2'].isin(['Paid Off', 'Return'])]

# loan age bands
loan_age_bins   = [0, 90, 180, 365, 730, 9999]
loan_age_labels = ['0-3 months', '3-6 months', '6-12 months', '1-2 years', '2+ years']
active_dec['loan_age_band'] = pd.cut(active_dec['CUSTOMER_AGE'], bins=loan_age_bins, labels=loan_age_labels, right=True)

# compute metrics by segment
seg = active_dec.groupby('loan_age_band', observed=True).agg(
    Loans    = ('LOAN_ID', 'count'),
    PAR30    = ('DAYS_PAST_DUE', lambda x: (x > 30).sum()),
    PAR90    = ('DAYS_PAST_DUE', lambda x: (x > 90).sum()),
    WriteOff = ('ACCOUNT_STATUS_L1', lambda x: (x == 'Write Off').sum()),
    AvgDPD   = ('DAYS_PAST_DUE', 'mean'),
)

seg['PAR30_Rate']    = seg['PAR30'] / seg['Loans']
seg['PAR90_Rate']    = seg['PAR90'] / seg['Loans']
seg['WriteOff_Rate'] = seg['WriteOff'] / seg['Loans']

print(seg[['Loans','PAR30_Rate','PAR90_Rate','WriteOff_Rate','AvgDPD']])

               Loans  PAR30_Rate  PAR90_Rate  WriteOff_Rate      AvgDPD
loan_age_band                                                          
0-3 months      3566    0.062535    0.000000       0.000000    4.699944
3-6 months      2502    0.239808    0.114309       0.040368   23.105516
6-12 months     3912    0.440951    0.338190       0.233129   73.268149
1-2 years       2942    0.998640    0.985724       0.706322  402.202923
2+ years         727    1.000000    1.000000       0.723521  701.640990


/tmp/ipykernel_8309/934103825.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  active_dec['loan_age_band'] = pd.cut(active_dec['CUSTOMER_AGE'], bins=loan_age_bins, labels=loan_age_labels, right=True)


**Question 2: Credit Outcomes × Customer Experience (35%)**

Using the NPS data, explore whether there is a relationship between credit performance (e.g. arrears,
default, account status) and customer satisfaction. Where do you see tension between collections
effectiveness and customer experience? Recommend one concrete action.

In [7]:
nps = pd.read_excel('./datasets/NPS Data.xlsx', index_col=False)
nps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4129 entries, 0 to 4128
Data columns (total 17 columns):
 #   Column                                                                                                                 Non-Null Count  Dtype         
---  ------                                                                                                                 --------------  -----         
 0   Submission ID                                                                                                          4129 non-null   object        
 1   Respondent ID                                                                                                          4129 non-null   object        
 2   Submitted at                                                                                                           4129 non-null   datetime64[ns]
 3   Loan Id                                                                                                          

In [11]:
# Load NPS
nps = pd.read_excel('./datasets/NPS Data.xlsx').dropna(how='all')
nps = nps.rename(columns={
    'Using a scale from 0 (not likely) to 10 (very likely), how likely are you to recommend MoPhones to friends or family?': 'nps_score',
    'Are you happy with the quality and performance of your MoPhones device?'   : 'happy_quality',
    'Are you happy with the service and support provided by MoPhones?'          : 'happy_service',
    'Have you ever experienced a delay in your payment reflecting in your Mophones account?' : 'payment_delay',
    'Have you ever had difficulty getting assistance from MoPhones customer support when needed?' : 'support_difficulty',
    'Have you ever had your phone lock despite making a payment on time?'        : 'phone_locked_despite_payment',
    'Loan Id': 'LOAN_ID'
})

# NPS category
nps['nps_category'] = pd.cut(
    nps['nps_score'],
    bins=[-1, 6, 8, 10],
    labels=['Detractor', 'Passive', 'Promoter']
)
nps['nps_category'].unique()

['Detractor', 'Promoter', 'Passive', NaN]
Categories (3, object): ['Detractor' < 'Passive' < 'Promoter']

In [12]:
# Overall NPS score
promoters  = (nps['nps_score'] >= 9).sum()
detractors = (nps['nps_score'] <= 6).sum()
total      = nps['nps_score'].notnull().sum()
print(f"Overall NPS: {((promoters - detractors)/total)*100:.1f}")

# Join to December credit snapshot
dec    = credit[credit['snapshot'] == 'dec'].copy()
merged = dec.merge(nps[['LOAN_ID','nps_score','nps_category',
                         'happy_quality','happy_service',
                         'payment_delay','support_difficulty',
                         'phone_locked_despite_payment']],
                   on='LOAN_ID', how='inner')

# NPS by account status
status_nps = merged.groupby('ACCOUNT_STATUS_L2', observed=True).agg(
    Count         = ('LOAN_ID', 'count'),
    Avg_NPS       = ('nps_score', 'mean'),
    Pct_Promoter  = ('nps_category', lambda x: (x=='Promoter').sum()/len(x)),
    Pct_Detractor = ('nps_category', lambda x: (x=='Detractor').sum()/len(x)),
).sort_values('Avg_NPS', ascending=False)
print(status_nps)

# NPS by DPD bucket
merged['dpd_bucket'] = pd.cut(
    merged['DAYS_PAST_DUE'],
    bins=[-1, 0, 30, 90, 9999],
    labels=['On Time','Early Arrears (1-30)','Serious Arrears (31-90)','Deep Default (90+)']
)

dpd_nps = merged.groupby('dpd_bucket', observed=True).agg(
    Count         = ('LOAN_ID', 'count'),
    Avg_NPS       = ('nps_score', 'mean'),
    Pct_Detractor = ('nps_category', lambda x: (x=='Detractor').sum()/len(x)),
    Avg_Arrears   = ('ARREARS', 'mean'),
).reset_index()
print(dpd_nps)

# Phone locked despite payment
lock_nps = merged.groupby('phone_locked_despite_payment', observed=True).agg(
    Count         = ('LOAN_ID','count'),
    Avg_NPS       = ('nps_score','mean'),
    Pct_Detractor = ('nps_category', lambda x: (x=='Detractor').sum()/len(x)),
).reset_index()
print(lock_nps)

Overall NPS: 5.9
                   Count   Avg_NPS  Pct_Promoter  Pct_Detractor
ACCOUNT_STATUS_L2                                              
FPD                   34  7.333333      0.529412       0.294118
PAR 7                161  7.307692      0.465839       0.298137
Active              1809  7.141304      0.427861       0.318961
Paid Off             848  7.120585      0.445755       0.316038
Inactive             289  6.820144      0.422145       0.363322
PAR 30               597  6.708772      0.408710       0.361809
FMD                   51  6.354167      0.450980       0.372549
Return               340  3.864048      0.208824       0.664706
                dpd_bucket  Count   Avg_NPS  Pct_Detractor   Avg_Arrears
0                  On Time   2663  7.128544       0.318813    469.415321
1     Early Arrears (1-30)    477  6.960870       0.343816   2096.765199
2  Serious Arrears (31-90)    305  7.129252       0.311475  10077.918033
3       Deep Default (90+)    684  5.123476       0

**Question 3: Data Quality & Recommendations (25%)**

What limitations did you encounter in the data? What is missing, inconsistent, or ambiguous? Propose
2–3 specific improvements to how MoPhones captures or structures credit data to make ongoing
monitoring easier.

**Data Quality Issues Found**

**Issue 1: CUSTOMER_AGE is mislabelled**

Column name suggests **customer age** in years but it actually stores loan age in days
Proven by 100% match with days since SALE_DATE
Max value is 1,056 — impossible as a human age

**Issue 2: Demographics cover only ~50% of the portfolio**

DOB data exists for 54% of loans
Income data exists for 51% of loans
Gender data exists for 51% of loans
NPS responses cover only 17% of loans
All demographic data depends on TransUnion bureau checks — customers who skipped or failed the check have no profile at all

**Issue 3:Invalid ages in DOB data**

61 customers recorded as under 18 — should not legally have credit
196 customers recorded as over 80 — highly suspicious
2 customers with negative ages — clear data entry errors

**Issue 4 :RETURN_DATE is corrupted in the December snapshot**

January to September stores recognisable dates e.g. 1/22/2025
December stores only 00:00.0 — a time fragment with no date
All return dates in the largest snapshot are unreadable

**Issue 5 :PAYMENT_AMOUNT and ADJUSTMENT_AMOUNT are almost entirely empty**

93.6% null in January, worsening to 99.9% null in December
Cannot reconstruct what a customer actually paid from these columns

**Issue 6:NPS survey has no timing link to credit events**

No field shows where the customer was in their loan lifecycle when they filled the survey
A customer surveyed right after a phone lock scores very differently from one surveyed during a smooth month
This makes the NPS-to-credit relationship ambiguous

**Issue 7 :ACCOUNT_STATUS_L1 has 20 overlapping labels**

Several labels are variations of the same concept e.g. First Month Default without inventory over 30, First Month Default without inventory 01-07, First Month Default without inventory 08-14
8 Demo records sitting in the live portfolio — likely test data contaminating metrics

**Issue 8 : Gender values are inconsistent**

Same gender stored as both Male and M, Female and F
Groups incorrectly into 5 categories instead of 2

**Issue 9 :Ghost column Unnamed: 28**

Completely empty column appearing in all 5 snapshots
Leftover from a spreadsheet export — adds noise with zero value


**3 Specific Improvements**
**Improvement 1:Make demographics mandatory at onboarding**

Capture DOB, gender, and declared monthly income as required fields at point of sale for every financed customer
Do not rely on bureau check as the only source of demographic data
This alone would take age and income band analysis from 50% portfolio coverage to 100%

**Improvement 2: Add credit context to every NPS survey**

At the moment a survey is sent, automatically record the customer's ACCOUNT_STATUS_L2 and DAYS_PAST_DUE alongside the response
This transforms NPS from a vague satisfaction score into a precise diagnostic tool that shows exactly how experience changes as customers move from active to arrears to collections

**Improvement 3: Enforce a schema validation check before every snapshot export**

Run an automated check before any data file leaves the source system covering: correct date formats, allowed status values only, gender standardised to two values, no empty ghost columns
This single fix would have caught Issues 4, 7, 8, and 9 before they reached the analyst